In [1]:
import os
from openpyxl import load_workbook

# Пути
INPUT_DIR = "puid2024"
HEADER_PATH = "header.xlsx"
OUTPUT_DIR = "puid2024_2608"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for highway in os.listdir(INPUT_DIR):
    highway_path = os.path.join(INPUT_DIR, highway)
    if not os.path.isdir(highway_path):
        continue

    output_highway_path = os.path.join(OUTPUT_DIR, highway)
    os.makedirs(output_highway_path, exist_ok=True)

    for file in os.listdir(highway_path):
        # Берём только .xlsx, игнорируем временные файлы Excel (~$) и сам шаблон
        if (
            not file.endswith(".xlsx")
            or file.startswith("~$")
            or file == os.path.basename(HEADER_PATH)
        ):
            continue

        xlsx_input_path = os.path.join(highway_path, file)
        xlsx_output_path = os.path.join(output_highway_path, file)

        # Загружаем копию шаблона
        wb = load_workbook(HEADER_PATH)
        ws = wb.active

        # Загружаем исходный XLSX (читаем только значения формул)
        src_wb = load_workbook(xlsx_input_path, data_only=True, read_only=True)
        src_ws = src_wb.active

        # Переносим строки (пропускаем полностью пустые)
        for row in src_ws.iter_rows(values_only=True):
            if any(cell is not None for cell in row):
                ws.append(row)

        wb.save(xlsx_output_path)
        print(f"✅ Сохранено: {xlsx_output_path}")